# Transistor Database Performance Dashboard

**Interactive overview of transistor database with live plotting and filtering**

This notebook provides:
1. Database overview with summary statistics
2. Interactive filtering by device type, voltage class, current rating, manufacturer, package
3. Live performance plots (switching energy, FOM, capacitance curves)
4. ChristenBielaModel analytical switching loss integration
5. Export capabilities for filtered lists and plots

**Prerequisites:**
```bash
pip install ipywidgets matplotlib plotly pandas numpy scipy
jupyter nbextension enable --py widgetsnbextension
```

## 1. Setup & Database Loading

In [ ]:
# Import required libraries
import sys
import os
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Circle
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Add transistordatabase to path
tdb_path = Path.cwd()
if str(tdb_path) not in sys.path:
    sys.path.insert(0, str(tdb_path))

# Import transistordatabase modules
from transistordatabase.core.repository import JsonTransistorRepository, JsonTransistorLoader
from transistordatabase.core.models import Transistor
from transistordatabase.analytical_models import (
    ChristenBielaModel,
    HalfBridgeParams,
    TransconductanceParams,
    calc_charge_equivalent_capacitance,
    calc_device_capacitances,
    fit_transconductance,
    get_package_inductance,
)

print("✓ Imports successful")
print(f"Working directory: {Path.cwd()}")

In [ ]:
# Load transistor database
DATABASE_DIR = Path.cwd() / "transistors_merged"

if not DATABASE_DIR.exists():
    print(f"⚠ Database directory not found: {DATABASE_DIR}")
    print("Please adjust DATABASE_DIR variable to point to your transistor JSON files.")
else:
    print(f"✓ Database directory found: {DATABASE_DIR}")

# Initialize repository and loader
repository = JsonTransistorRepository(DATABASE_DIR)
loader = JsonTransistorLoader()

# Load all transistors
transistor_names = repository.list_all()
print(f"\n📊 Found {len(transistor_names)} transistors in database")

# Load all transistor objects (with error handling)
transistors = {}
load_errors = []

print("\nLoading transistors...")
for i, name in enumerate(transistor_names):
    try:
        transistors[name] = repository.get_by_name(name)
        if (i + 1) % 50 == 0:
            print(f"  Loaded {i + 1}/{len(transistor_names)}...")
    except Exception as e:
        load_errors.append((name, str(e)))

print(f"\n✓ Successfully loaded {len(transistors)} transistors")
if load_errors:
    print(f"⚠ Failed to load {len(load_errors)} transistors (see load_errors list)")

## 2. Database Summary Statistics

In [ ]:
# Extract summary data from all transistors
summary_data = []

for name, t in transistors.items():
    # Extract key metadata
    device_type = t.metadata.type if t.metadata else "Unknown"
    manufacturer = t.metadata.manufacturer if t.metadata else "Unknown"
    housing = t.metadata.housing_type if t.metadata else "Unknown"
    
    # Electrical ratings
    v_max = t.electrical_ratings.v_abs_max if t.electrical_ratings else 0.0
    i_max = t.electrical_ratings.i_abs_max if t.electrical_ratings else 0.0
    i_cont = t.electrical_ratings.i_cont if t.electrical_ratings else 0.0
    
    # Data completeness flags
    has_switch = t.switch is not None
    has_diode = t.diode is not None
    has_channel_sw = len(t.switch.channel_data) > 0 if has_switch else False
    has_channel_di = len(t.diode.channel_data) > 0 if has_diode else False
    has_e_on = len(t.switch.e_on_data) > 0 if has_switch else False
    has_e_off = len(t.switch.e_off_data) > 0 if has_switch else False
    has_e_rr = len(t.diode.e_rr_data) > 0 if has_diode else False
    has_c_oss = len(t.c_oss) > 0 if t.c_oss else False
    has_c_iss = len(t.c_iss) > 0 if t.c_iss else False
    has_c_rss = len(t.c_rss) > 0 if t.c_rss else False
    has_gate_charge = len(t.switch.gate_charge_curves) > 0 if has_switch else False
    
    # Extract typical R_ds(on) from channel data (at lowest voltage if available)
    r_ds_on = None
    if has_channel_sw and len(t.switch.channel_data[0].graph_v_i[0]) > 0:
        ch = t.switch.channel_data[0]
        v_ds = ch.graph_v_i[0]
        i_ds = ch.graph_v_i[1]
        # Linear region: R = V/I at low V_ds
        idx_low = np.where((v_ds > 0.1) & (v_ds < 2.0) & (i_ds > 1.0))[0]
        if len(idx_low) > 0:
            r_ds_on = float(np.median(v_ds[idx_low] / i_ds[idx_low]))
    
    # Extract typical Q_g from gate charge curve (total charge at V_gs=15V)
    q_g = None
    if has_gate_charge:
        gc = t.switch.gate_charge_curves[0]
        if len(gc.graph_q_v[0]) > 0:
            q_arr = gc.graph_q_v[0]
            v_arr = gc.graph_q_v[1]
            # Find charge at V_gs ~ 15V or max
            idx_15v = np.where(v_arr >= 14.0)[0]
            if len(idx_15v) > 0:
                q_g = float(q_arr[idx_15v[0]])
            else:
                q_g = float(q_arr[-1])  # Use max charge
    
    summary_data.append({
        'name': name,
        'type': device_type,
        'manufacturer': manufacturer,
        'housing': housing,
        'v_max': v_max,
        'i_max': i_max,
        'i_cont': i_cont,
        'r_ds_on': r_ds_on,
        'q_g': q_g,
        'fom': r_ds_on * q_g if (r_ds_on and q_g) else None,
        'has_channel_sw': has_channel_sw,
        'has_channel_di': has_channel_di,
        'has_e_on': has_e_on,
        'has_e_off': has_e_off,
        'has_e_rr': has_e_rr,
        'has_c_oss': has_c_oss,
        'has_c_iss': has_c_iss,
        'has_c_rss': has_c_rss,
        'has_gate_charge': has_gate_charge,
    })

df = pd.DataFrame(summary_data)
print(f"✓ Extracted summary data for {len(df)} devices\n")

# Display summary statistics
print("=" * 70)
print("DATABASE SUMMARY STATISTICS")
print("=" * 70)

print(f"\n📊 Total Devices: {len(df)}")
print(f"\n🔌 Device Types:")
print(df['type'].value_counts().to_string())

print(f"\n🏭 Top 10 Manufacturers:")
print(df['manufacturer'].value_counts().head(10).to_string())

print(f"\n📦 Top 10 Package Types:")
print(df['housing'].value_counts().head(10).to_string())

print(f"\n⚡ Voltage Classes (V_max):")
v_classes = pd.cut(df[df['v_max'] > 0]['v_max'], 
                   bins=[0, 100, 300, 600, 900, 1200, 1700, 3500, 10000],
                   labels=['<100V', '100-300V', '300-600V', '600-900V', 
                          '900-1200V', '1200-1700V', '1700-3500V', '>3500V'])
print(v_classes.value_counts().sort_index().to_string())

print(f"\n📈 Data Completeness:")
print(f"  Switch channel data:    {df['has_channel_sw'].sum():4d} ({100*df['has_channel_sw'].mean():.1f}%)")
print(f"  Diode channel data:     {df['has_channel_di'].sum():4d} ({100*df['has_channel_di'].mean():.1f}%)")
print(f"  E_on data:              {df['has_e_on'].sum():4d} ({100*df['has_e_on'].mean():.1f}%)")
print(f"  E_off data:             {df['has_e_off'].sum():4d} ({100*df['has_e_off'].mean():.1f}%)")
print(f"  E_rr data:              {df['has_e_rr'].sum():4d} ({100*df['has_e_rr'].mean():.1f}%)")
print(f"  C_oss curves:           {df['has_c_oss'].sum():4d} ({100*df['has_c_oss'].mean():.1f}%)")
print(f"  C_iss curves:           {df['has_c_iss'].sum():4d} ({100*df['has_c_iss'].mean():.1f}%)")
print(f"  C_rss curves:           {df['has_c_rss'].sum():4d} ({100*df['has_c_rss'].mean():.1f}%)")
print(f"  Gate charge curves:     {df['has_gate_charge'].sum():4d} ({100*df['has_gate_charge'].mean():.1f}%)")

print(f"\n⚙️  Performance Metrics (where available):")
print(f"  R_ds(on) available:     {df['r_ds_on'].notna().sum():4d} devices")
if df['r_ds_on'].notna().sum() > 0:
    print(f"    Range: {df['r_ds_on'].min()*1000:.2f} - {df['r_ds_on'].max()*1000:.2f} mΩ")
    print(f"    Median: {df['r_ds_on'].median()*1000:.2f} mΩ")
print(f"  Q_g available:          {df['q_g'].notna().sum():4d} devices")
if df['q_g'].notna().sum() > 0:
    print(f"    Range: {df['q_g'].min()*1e9:.1f} - {df['q_g'].max()*1e9:.1f} nC")
    print(f"    Median: {df['q_g'].median()*1e9:.1f} nC")
print(f"  FOM (R*Q_g) available:  {df['fom'].notna().sum():4d} devices")
if df['fom'].notna().sum() > 0:
    print(f"    Median FOM: {df['fom'].median()*1e12:.1f} mΩ·nC")

print("\n" + "=" * 70)

## 3. Interactive Filtering Controls

Use the widgets below to filter devices by type, voltage class, current rating, manufacturer, and data completeness.

In [ ]:
# Create interactive filter widgets
style = {'description_width': '150px'}
layout = widgets.Layout(width='500px')

# Device type filter
device_types = ['All'] + sorted(df['type'].unique().tolist())
type_filter = widgets.SelectMultiple(
    options=device_types,
    value=['All'],
    description='Device Type:',
    style=style,
    layout=layout
)

# Voltage class filter
v_min_filter = widgets.FloatSlider(
    value=0,
    min=0,
    max=10000,
    step=100,
    description='Min Voltage (V):',
    style=style,
    layout=layout
)

v_max_filter = widgets.FloatSlider(
    value=10000,
    min=0,
    max=10000,
    step=100,
    description='Max Voltage (V):',
    style=style,
    layout=layout
)

# Current rating filter
i_min_filter = widgets.FloatSlider(
    value=0,
    min=0,
    max=1000,
    step=10,
    description='Min Current (A):',
    style=style,
    layout=layout
)

# Manufacturer filter
manufacturers = ['All'] + sorted([m for m in df['manufacturer'].unique() if m != 'Unknown'])
mfr_filter = widgets.SelectMultiple(
    options=manufacturers,
    value=['All'],
    description='Manufacturer:',
    style=style,
    layout=layout
)

# Data completeness filters
require_channel = widgets.Checkbox(
    value=False,
    description='Require channel data',
    style=style
)

require_switching = widgets.Checkbox(
    value=False,
    description='Require switching loss data',
    style=style
)

require_capacitance = widgets.Checkbox(
    value=False,
    description='Require capacitance curves',
    style=style
)

require_gate_charge = widgets.Checkbox(
    value=False,
    description='Require gate charge curves',
    style=style
)

# Filter button
filter_button = widgets.Button(
    description='Apply Filters',
    button_style='primary',
    icon='filter'
)

# Output widget for filtered results
filter_output = widgets.Output()

# Global variable to store filtered dataframe
df_filtered = df.copy()

def apply_filters(b):
    global df_filtered
    with filter_output:
        clear_output(wait=True)
        
        # Start with full dataset
        df_filtered = df.copy()
        
        # Apply device type filter
        if 'All' not in type_filter.value:
            df_filtered = df_filtered[df_filtered['type'].isin(type_filter.value)]
        
        # Apply voltage filter
        df_filtered = df_filtered[
            (df_filtered['v_max'] >= v_min_filter.value) & 
            (df_filtered['v_max'] <= v_max_filter.value)
        ]
        
        # Apply current filter
        df_filtered = df_filtered[df_filtered['i_max'] >= i_min_filter.value]
        
        # Apply manufacturer filter
        if 'All' not in mfr_filter.value:
            df_filtered = df_filtered[df_filtered['manufacturer'].isin(mfr_filter.value)]
        
        # Apply data completeness filters
        if require_channel.value:
            df_filtered = df_filtered[df_filtered['has_channel_sw']]
        
        if require_switching.value:
            df_filtered = df_filtered[df_filtered['has_e_on'] | df_filtered['has_e_off']]
        
        if require_capacitance.value:
            df_filtered = df_filtered[df_filtered['has_c_oss'] | df_filtered['has_c_iss']]
        
        if require_gate_charge.value:
            df_filtered = df_filtered[df_filtered['has_gate_charge']]
        
        # Display results
        print(f"✓ Filtered to {len(df_filtered)} devices (from {len(df)} total)")
        print(f"\nDevice type breakdown:")
        print(df_filtered['type'].value_counts().to_string())
        print(f"\nTop manufacturers:")
        print(df_filtered['manufacturer'].value_counts().head(5).to_string())

filter_button.on_click(apply_filters)

# Display all filter widgets
print("Configure filters and click 'Apply Filters' to update dataset:\n")
display(widgets.VBox([
    widgets.HTML("<h3>Filter Controls</h3>"),
    type_filter,
    v_min_filter,
    v_max_filter,
    i_min_filter,
    mfr_filter,
    widgets.HTML("<h4>Data Completeness Requirements</h4>"),
    require_channel,
    require_switching,
    require_capacitance,
    require_gate_charge,
    filter_button,
    filter_output
]))

## 4. Performance Plots

### 4.1 Figure of Merit Analysis (R_ds × Q_g)

In [ ]:
# FOM scatter plot with interactive plotly
df_plot = df_filtered[df_filtered['fom'].notna()].copy()

if len(df_plot) == 0:
    print("⚠ No devices with both R_ds(on) and Q_g data in filtered set")
else:
    # Convert to display units
    df_plot['R_ds_mohm'] = df_plot['r_ds_on'] * 1000  # mΩ
    df_plot['Q_g_nC'] = df_plot['q_g'] * 1e9  # nC
    df_plot['FOM'] = df_plot['fom'] * 1e12  # mΩ·nC
    
    fig = px.scatter(
        df_plot,
        x='Q_g_nC',
        y='R_ds_mohm',
        color='type',
        size='v_max',
        hover_data=['name', 'manufacturer', 'v_max', 'FOM'],
        log_x=True,
        log_y=True,
        title=f'Figure of Merit: R_ds(on) vs Q_g ({len(df_plot)} devices)',
        labels={'Q_g_nC': 'Gate Charge Q_g (nC)', 'R_ds_mohm': 'R_ds(on) (mΩ)'},
        width=1000,
        height=600
    )
    
    # Add constant FOM contour lines
    fom_levels = [10, 50, 100, 500, 1000, 5000]  # mΩ·nC
    q_range = np.logspace(np.log10(df_plot['Q_g_nC'].min()), 
                          np.log10(df_plot['Q_g_nC'].max()), 100)
    
    for fom_val in fom_levels:
        r_contour = fom_val / q_range
        fig.add_trace(go.Scatter(
            x=q_range,
            y=r_contour,
            mode='lines',
            line=dict(dash='dash', color='gray', width=1),
            name=f'FOM={fom_val} mΩ·nC',
            showlegend=True
        ))
    
    fig.update_layout(
        xaxis_title='Gate Charge Q_g (nC)',
        yaxis_title='R_ds(on) (mΩ)',
        hovermode='closest'
    )
    
    fig.show()
    
    print(f"\n✓ Plotted {len(df_plot)} devices with FOM data")
    print(f"  Best FOM: {df_plot['FOM'].min():.1f} mΩ·nC ({df_plot.loc[df_plot['FOM'].idxmin(), 'name']})")
    print(f"  Worst FOM: {df_plot['FOM'].max():.1f} mΩ·nC ({df_plot.loc[df_plot['FOM'].idxmax(), 'name']})")

### 4.2 Voltage vs Current Rating Distribution

In [ ]:
# Voltage-current scatter with device types
df_plot = df_filtered[(df_filtered['v_max'] > 0) & (df_filtered['i_max'] > 0)].copy()

if len(df_plot) == 0:
    print("⚠ No devices with voltage and current ratings in filtered set")
else:
    fig = px.scatter(
        df_plot,
        x='v_max',
        y='i_max',
        color='type',
        size='fom',
        hover_data=['name', 'manufacturer', 'housing'],
        title=f'Voltage vs Current Rating ({len(df_plot)} devices)',
        labels={'v_max': 'Max Voltage V_ds (V)', 'i_max': 'Max Current I_d (A)'},
        width=1000,
        height=600,
        log_x=True,
        log_y=True
    )
    
    fig.update_layout(hovermode='closest')
    fig.show()
    
    print(f"\n✓ Plotted {len(df_plot)} devices")

### 4.3 Capacitance Curves Overlay

Select devices to overlay their C_oss(V), C_iss(V), and C_rss(V) curves.

In [ ]:
# Device selector for capacitance overlay
devices_with_cap = df_filtered[df_filtered['has_c_oss']]['name'].tolist()

if len(devices_with_cap) == 0:
    print("⚠ No devices with capacitance curves in filtered set")
else:
    cap_device_selector = widgets.SelectMultiple(
        options=devices_with_cap[:50],  # Limit to first 50 for performance
        value=[devices_with_cap[0]] if devices_with_cap else [],
        description='Select Devices:',
        style={'description_width': '150px'},
        layout=widgets.Layout(width='600px', height='200px')
    )
    
    cap_plot_button = widgets.Button(
        description='Plot Capacitance',
        button_style='success',
        icon='line-chart'
    )
    
    cap_output = widgets.Output()
    
    def plot_capacitance(b):
        with cap_output:
            clear_output(wait=True)
            
            if len(cap_device_selector.value) == 0:
                print("⚠ Please select at least one device")
                return
            
            # Create subplots for C_oss, C_iss, C_rss
            fig = make_subplots(
                rows=1, cols=3,
                subplot_titles=('C_oss(V)', 'C_iss(V)', 'C_rss(V)'),
                horizontal_spacing=0.1
            )
            
            colors = px.colors.qualitative.Plotly
            
            for idx, dev_name in enumerate(cap_device_selector.value):
                t = transistors[dev_name]
                color = colors[idx % len(colors)]
                
                # Plot C_oss
                if t.c_oss and len(t.c_oss) > 0:
                    for c_oss_curve in t.c_oss:
                        v_data = c_oss_curve.graph_v_c[0]
                        c_data = c_oss_curve.graph_v_c[1] * 1e12  # Convert to pF
                        fig.add_trace(
                            go.Scatter(x=v_data, y=c_data, mode='lines', 
                                      name=dev_name, line=dict(color=color),
                                      showlegend=(idx==0)),
                            row=1, col=1
                        )
                
                # Plot C_iss
                if t.c_iss and len(t.c_iss) > 0:
                    for c_iss_curve in t.c_iss:
                        v_data = c_iss_curve.graph_v_c[0]
                        c_data = c_iss_curve.graph_v_c[1] * 1e12
                        fig.add_trace(
                            go.Scatter(x=v_data, y=c_data, mode='lines',
                                      name=dev_name, line=dict(color=color),
                                      showlegend=False),
                            row=1, col=2
                        )
                
                # Plot C_rss
                if t.c_rss and len(t.c_rss) > 0:
                    for c_rss_curve in t.c_rss:
                        v_data = c_rss_curve.graph_v_c[0]
                        c_data = c_rss_curve.graph_v_c[1] * 1e12
                        fig.add_trace(
                            go.Scatter(x=v_data, y=c_data, mode='lines',
                                      name=dev_name, line=dict(color=color),
                                      showlegend=False),
                            row=1, col=3
                        )
            
            fig.update_xaxes(title_text='Voltage (V)', type='log', row=1, col=1)
            fig.update_xaxes(title_text='Voltage (V)', type='log', row=1, col=2)
            fig.update_xaxes(title_text='Voltage (V)', type='log', row=1, col=3)
            
            fig.update_yaxes(title_text='Capacitance (pF)', type='log', row=1, col=1)
            fig.update_yaxes(title_text='Capacitance (pF)', type='log', row=1, col=2)
            fig.update_yaxes(title_text='Capacitance (pF)', type='log', row=1, col=3)
            
            fig.update_layout(
                title_text=f'Capacitance Curves Comparison ({len(cap_device_selector.value)} devices)',
                width=1400,
                height=500,
                showlegend=True
            )
            
            fig.show()
            print(f"\n✓ Plotted capacitance curves for {len(cap_device_selector.value)} devices")
    
    cap_plot_button.on_click(plot_capacitance)
    
    display(widgets.VBox([
        widgets.HTML("<h4>Select devices to overlay capacitance curves:</h4>"),
        cap_device_selector,
        cap_plot_button,
        cap_output
    ]))

### 4.4 Channel Characteristics (I-V Curves)

In [ ]:
# Device selector for I-V curves
devices_with_channel = df_filtered[df_filtered['has_channel_sw']]['name'].tolist()

if len(devices_with_channel) == 0:
    print("⚠ No devices with channel characteristics in filtered set")
else:
    iv_device_selector = widgets.Dropdown(
        options=devices_with_channel[:100],
        value=devices_with_channel[0] if devices_with_channel else None,
        description='Device:',
        style={'description_width': '150px'},
        layout=widgets.Layout(width='600px')
    )
    
    iv_plot_button = widgets.Button(
        description='Plot I-V Curves',
        button_style='info',
        icon='line-chart'
    )
    
    iv_output = widgets.Output()
    
    def plot_iv_curves(b):
        with iv_output:
            clear_output(wait=True)
            
            dev_name = iv_device_selector.value
            t = transistors[dev_name]
            
            fig = go.Figure()
            
            for ch_data in t.switch.channel_data:
                v_ds = ch_data.graph_v_i[0]
                i_ds = ch_data.graph_v_i[1]
                v_g = ch_data.v_g if ch_data.v_g else "?"
                temp = ch_data.t_j if ch_data.t_j else "?"
                
                fig.add_trace(go.Scatter(
                    x=v_ds,
                    y=i_ds,
                    mode='lines',
                    name=f'V_gs={v_g}V, T_j={temp}°C'
                ))
            
            fig.update_layout(
                title=f'Channel Characteristics: {dev_name}',
                xaxis_title='V_ds (V)',
                yaxis_title='I_ds (A)',
                width=1000,
                height=600,
                hovermode='x unified'
            )
            
            fig.show()
            
            # Print device info
            print(f"\nDevice: {dev_name}")
            print(f"  Type: {t.metadata.type}")
            print(f"  Manufacturer: {t.metadata.manufacturer}")
            print(f"  V_max: {t.electrical_ratings.v_abs_max} V")
            print(f"  I_max: {t.electrical_ratings.i_abs_max} A")
            print(f"  Number of curves: {len(t.switch.channel_data)}")
    
    iv_plot_button.on_click(plot_iv_curves)
    
    display(widgets.VBox([
        widgets.HTML("<h4>Select device to view channel characteristics:</h4>"),
        iv_device_selector,
        iv_plot_button,
        iv_output
    ]))

## 5. ChristenBielaModel Integration

Run analytical switching loss calculations using the Christen-Biela model and compare with measured data (if available).

In [ ]:
# Operating conditions for Christen-Biela model
style = {'description_width': '200px'}
layout = widgets.Layout(width='400px')

cb_v_dc = widgets.FloatText(value=600.0, description='DC Bus Voltage (V):', style=style, layout=layout)
cb_i_load = widgets.FloatText(value=20.0, description='Load Current (A):', style=style, layout=layout)
cb_v_g_on = widgets.FloatText(value=20.0, description='V_g,on (V):', style=style, layout=layout)
cb_v_g_off = widgets.FloatText(value=-5.0, description='V_g,off (V):', style=style, layout=layout)
cb_r_g = widgets.FloatText(value=10.0, description='Gate Resistance (Ω):', style=style, layout=layout)

cb_device_selector = widgets.Dropdown(
    options=df_filtered[df_filtered['has_c_oss'] & df_filtered['has_channel_sw']]['name'].tolist()[:50],
    description='Device:',
    style=style,
    layout=layout
)

cb_run_button = widgets.Button(
    description='Run Analytical Model',
    button_style='warning',
    icon='calculator'
)

cb_output = widgets.Output()

def run_christen_biela(b):
    with cb_output:
        clear_output(wait=True)
        
        dev_name = cb_device_selector.value
        if not dev_name:
            print("⚠ Please select a device")
            return
        
        t = transistors[dev_name]
        
        print("=" * 70)
        print(f"CHRISTEN-BIELA MODEL: {dev_name}")
        print("=" * 70)
        
        try:
            # Extract capacitances
            if not t.c_oss or len(t.c_oss) == 0:
                print("⚠ Device missing C_oss data")
                return
            
            v_0 = cb_v_dc.value
            
            # Get C(V) curves
            c_oss_curve = t.c_oss[0]
            v_oss = c_oss_curve.graph_v_c[0]
            c_oss_vals = c_oss_curve.graph_v_c[1]
            
            c_oss_eq = calc_charge_equivalent_capacitance(v_oss, c_oss_vals, v_0)
            
            # Try to get C_iss and C_rss
            if t.c_iss and len(t.c_iss) > 0:
                c_iss_curve = t.c_iss[0]
                v_iss = c_iss_curve.graph_v_c[0]
                c_iss_vals = c_iss_curve.graph_v_c[1]
                c_iss_eq = calc_charge_equivalent_capacitance(v_iss, c_iss_vals, v_0)
            else:
                c_iss_eq = 2.0 * c_oss_eq  # Estimate
            
            if t.c_rss and len(t.c_rss) > 0:
                c_rss_curve = t.c_rss[0]
                v_rss = c_rss_curve.graph_v_c[0]
                c_rss_vals = c_rss_curve.graph_v_c[1]
                c_rss_eq = calc_charge_equivalent_capacitance(v_rss, c_rss_vals, v_0)
            else:
                c_rss_eq = 0.3 * c_oss_eq  # Estimate
            
            # Calculate device capacitances
            c_gs, c_ds, c_gd = calc_device_capacitances(c_iss_eq, c_oss_eq, c_rss_eq)
            
            # Calculate Q_oss
            q_oss = np.trapezoid(c_oss_vals, v_oss)
            
            print(f"\n[Capacitances at V_0={v_0}V]")
            print(f"  C_oss_eq = {c_oss_eq*1e12:.2f} pF")
            print(f"  C_iss_eq = {c_iss_eq*1e12:.2f} pF")
            print(f"  C_rss_eq = {c_rss_eq*1e12:.2f} pF")
            print(f"  C_gs = {c_gs*1e12:.2f} pF")
            print(f"  C_ds = {c_ds*1e12:.2f} pF")
            print(f"  C_gd = {c_gd*1e12:.2f} pF")
            print(f"  Q_oss = {q_oss*1e9:.2f} nC")
            
            # Fit transconductance model
            if not t.switch.channel_data or len(t.switch.channel_data) < 3:
                print("\n⚠ Insufficient channel data for transconductance fitting (need ≥3 V_gs points)")
                print("  Using default transconductance parameters")
                gm_params = TransconductanceParams(k_1=0.5, k_2=0.0, x=2.0, v_th=3.5)
            else:
                try:
                    gm_params = fit_transconductance(t.switch.channel_data)
                    print(f"\n[Transconductance Model]")
                    print(f"  k_1 = {gm_params.k_1:.3f}")
                    print(f"  k_2 = {gm_params.k_2:.3f}")
                    print(f"  x = {gm_params.x:.3f}")
                    print(f"  V_th = {gm_params.v_th:.2f} V")
                except Exception as e:
                    print(f"\n⚠ Transconductance fitting failed: {e}")
                    print("  Using default parameters")
                    gm_params = TransconductanceParams(k_1=0.5, k_2=0.0, x=2.0, v_th=3.5)
            
            # Get package inductance
            l_s, l_d = get_package_inductance(t.metadata.housing_type)
            
            # Create Christen-Biela model
            cb_model = ChristenBielaModel(
                c_gs=c_gs,
                c_ds=c_ds,
                c_gd=c_gd,
                q_oss=q_oss,
                gm_params=gm_params,
                rr_params=None  # Skip reverse recovery for now
            )
            
            # Operating conditions
            params = HalfBridgeParams(
                v_0=cb_v_dc.value,
                i_0=cb_i_load.value,
                v_g_on=cb_v_g_on.value,
                v_g_off=cb_v_g_off.value,
                r_g=cb_r_g.value,
                l_s=l_s,
                l_d=l_d
            )
            
            print(f"\n[Operating Conditions]")
            print(f"  V_0 = {params.v_0:.0f} V")
            print(f"  I_0 = {params.i_0:.1f} A")
            print(f"  V_g,on = {params.v_g_on:.0f} V")
            print(f"  V_g,off = {params.v_g_off:.0f} V")
            print(f"  R_g = {params.r_g:.1f} Ω")
            print(f"  L_s = {params.l_s*1e9:.1f} nH (package: {t.metadata.housing_type})")
            print(f"  L_d = {params.l_d*1e9:.1f} nH")
            
            # Calculate turn-on energy
            result_on = cb_model.calc_turn_on_energy(params)
            
            print(f"\n[Turn-On Results]")
            print(f"  E_on = {result_on['e_on']*1e6:.2f} µJ")
            print(f"  I_oss = {result_on['i_oss']:.2f} A")
            print(f"  V_mil = {result_on['v_mil']:.2f} V")
            print(f"  t_ri = {result_on['t_ri']*1e9:.2f} ns (current rise)")
            print(f"  t_fv = {result_on['t_fv']*1e9:.2f} ns (voltage fall)")
            print(f"  g_m,1b = {result_on['gm_1b']:.3f} S")
            print(f"  g_m,3b = {result_on['gm_3b']:.3f} S")
            
            # Calculate turn-off energy
            result_off = cb_model.calc_turn_off_energy(params)
            
            print(f"\n[Turn-Off Results]")
            print(f"  E_off = {result_off['e_off']*1e6:.2f} µJ")
            print(f"  I_oss = {result_off['i_oss']:.2f} A")
            print(f"  V_mil = {result_off['v_mil']:.2f} V")
            print(f"  t_rv = {result_off['t_rv']*1e9:.2f} ns (voltage rise)")
            print(f"  t_fi = {result_off['t_fi']*1e9:.2f} ns (current fall)")
            print(f"  g_m = {result_off['gm']:.3f} S")
            
            # ZVS boundary
            i_0_zvs = cb_model.calc_i_0_zvs(params)
            print(f"\n[ZVS Boundary]")
            print(f"  I_0,ZVS = {i_0_zvs:.2f} A")
            if params.i_0 < i_0_zvs:
                print(f"  ✓ ZVS condition (I_0 < I_0,ZVS): Lossless turn-off possible")
            else:
                print(f"  ✗ Hard switching (I_0 > I_0,ZVS): Losses occur")
            
            # Total energy
            e_total = result_on['e_on'] + result_off['e_off']
            print(f"\n[Total Switching Energy]")
            print(f"  E_total = {e_total*1e6:.2f} µJ/cycle")
            print(f"  E_on / E_total = {100*result_on['e_on']/e_total:.1f}%")
            print(f"  E_off / E_total = {100*result_off['e_off']/e_total:.1f}%")
            
            # Compare with measured data if available
            if t.switch.e_on_data or t.switch.e_off_data:
                print(f"\n[Comparison with Measured Data]")
                
                if t.switch.e_on_data:
                    # Find closest measured point
                    for e_data in t.switch.e_on_data:
                        if hasattr(e_data, 'dataset_type') and e_data.dataset_type == 'graph_i_e':
                            i_vals = e_data.graph_i_e[0]
                            e_vals = e_data.graph_i_e[1]
                            e_meas = np.interp(params.i_0, i_vals, e_vals)
                            error = 100 * (result_on['e_on'] - e_meas) / e_meas if e_meas > 0 else 0
                            print(f"  E_on: Measured={e_meas*1e6:.2f} µJ, Model={result_on['e_on']*1e6:.2f} µJ, Error={error:+.1f}%")
                            break
                
                if t.switch.e_off_data:
                    for e_data in t.switch.e_off_data:
                        if hasattr(e_data, 'dataset_type') and e_data.dataset_type == 'graph_i_e':
                            i_vals = e_data.graph_i_e[0]
                            e_vals = e_data.graph_i_e[1]
                            e_meas = np.interp(params.i_0, i_vals, e_vals)
                            error = 100 * (result_off['e_off'] - e_meas) / e_meas if e_meas > 0 else 0
                            print(f"  E_off: Measured={e_meas*1e6:.2f} µJ, Model={result_off['e_off']*1e6:.2f} µJ, Error={error:+.1f}%")
                            break
            
            print("\n" + "=" * 70)
            
        except Exception as e:
            print(f"\n⚠ Error running Christen-Biela model: {e}")
            import traceback
            traceback.print_exc()

cb_run_button.on_click(run_christen_biela)

display(widgets.VBox([
    widgets.HTML("<h3>Christen-Biela Analytical Model</h3>"),
    widgets.HTML("<p>Configure operating conditions and select a device with capacitance and channel data.</p>"),
    cb_v_dc,
    cb_i_load,
    cb_v_g_on,
    cb_v_g_off,
    cb_r_g,
    cb_device_selector,
    cb_run_button,
    cb_output
]))

### 5.1 Energy vs Current Sweep

Run analytical model over a range of load currents to generate E(I) curves.

In [ ]:
# Current sweep widget
sweep_i_min = widgets.FloatText(value=5.0, description='I_min (A):', style=style, layout=layout)
sweep_i_max = widgets.FloatText(value=50.0, description='I_max (A):', style=style, layout=layout)
sweep_n_points = widgets.IntText(value=20, description='Number of points:', style=style, layout=layout)

sweep_device_selector = widgets.Dropdown(
    options=df_filtered[df_filtered['has_c_oss'] & df_filtered['has_channel_sw']]['name'].tolist()[:50],
    description='Device:',
    style=style,
    layout=layout
)

sweep_run_button = widgets.Button(
    description='Run Current Sweep',
    button_style='danger',
    icon='line-chart'
)

sweep_output = widgets.Output()

def run_current_sweep(b):
    with sweep_output:
        clear_output(wait=True)
        
        dev_name = sweep_device_selector.value
        if not dev_name:
            print("⚠ Please select a device")
            return
        
        t = transistors[dev_name]
        
        print(f"Running current sweep for {dev_name}...")
        
        try:
            # Setup model (same as before)
            v_0 = cb_v_dc.value
            c_oss_curve = t.c_oss[0]
            v_oss = c_oss_curve.graph_v_c[0]
            c_oss_vals = c_oss_curve.graph_v_c[1]
            c_oss_eq = calc_charge_equivalent_capacitance(v_oss, c_oss_vals, v_0)
            
            if t.c_iss and len(t.c_iss) > 0:
                c_iss_curve = t.c_iss[0]
                c_iss_eq = calc_charge_equivalent_capacitance(c_iss_curve.graph_v_c[0], c_iss_curve.graph_v_c[1], v_0)
            else:
                c_iss_eq = 2.0 * c_oss_eq
            
            if t.c_rss and len(t.c_rss) > 0:
                c_rss_curve = t.c_rss[0]
                c_rss_eq = calc_charge_equivalent_capacitance(c_rss_curve.graph_v_c[0], c_rss_curve.graph_v_c[1], v_0)
            else:
                c_rss_eq = 0.3 * c_oss_eq
            
            c_gs, c_ds, c_gd = calc_device_capacitances(c_iss_eq, c_oss_eq, c_rss_eq)
            q_oss = np.trapezoid(c_oss_vals, v_oss)
            
            if len(t.switch.channel_data) >= 3:
                gm_params = fit_transconductance(t.switch.channel_data)
            else:
                gm_params = TransconductanceParams(k_1=0.5, k_2=0.0, x=2.0, v_th=3.5)
            
            l_s, l_d = get_package_inductance(t.metadata.housing_type)
            
            cb_model = ChristenBielaModel(
                c_gs=c_gs, c_ds=c_ds, c_gd=c_gd, q_oss=q_oss,
                gm_params=gm_params, rr_params=None
            )
            
            # Current sweep
            i_range = np.linspace(sweep_i_min.value, sweep_i_max.value, sweep_n_points.value)
            e_on_list = []
            e_off_list = []
            
            for i_val in i_range:
                params = HalfBridgeParams(
                    v_0=cb_v_dc.value, i_0=i_val,
                    v_g_on=cb_v_g_on.value, v_g_off=cb_v_g_off.value,
                    r_g=cb_r_g.value, l_s=l_s, l_d=l_d
                )
                
                result_on = cb_model.calc_turn_on_energy(params)
                result_off = cb_model.calc_turn_off_energy(params)
                
                e_on_list.append(result_on['e_on'])
                e_off_list.append(result_off['e_off'])
            
            # Plot results
            fig = go.Figure()
            
            fig.add_trace(go.Scatter(
                x=i_range, y=np.array(e_on_list)*1e6,
                mode='lines+markers', name='E_on (model)',
                line=dict(color='blue', width=2)
            ))
            
            fig.add_trace(go.Scatter(
                x=i_range, y=np.array(e_off_list)*1e6,
                mode='lines+markers', name='E_off (model)',
                line=dict(color='red', width=2)
            ))
            
            fig.add_trace(go.Scatter(
                x=i_range, y=(np.array(e_on_list) + np.array(e_off_list))*1e6,
                mode='lines+markers', name='E_total (model)',
                line=dict(color='green', width=2)
            ))
            
            # Overlay measured data if available
            if t.switch.e_on_data:
                for e_data in t.switch.e_on_data:
                    if hasattr(e_data, 'dataset_type') and e_data.dataset_type == 'graph_i_e':
                        fig.add_trace(go.Scatter(
                            x=e_data.graph_i_e[0], y=e_data.graph_i_e[1]*1e6,
                            mode='markers', name='E_on (measured)',
                            marker=dict(symbol='square', size=8, color='blue')
                        ))
            
            if t.switch.e_off_data:
                for e_data in t.switch.e_off_data:
                    if hasattr(e_data, 'dataset_type') and e_data.dataset_type == 'graph_i_e':
                        fig.add_trace(go.Scatter(
                            x=e_data.graph_i_e[0], y=e_data.graph_i_e[1]*1e6,
                            mode='markers', name='E_off (measured)',
                            marker=dict(symbol='square', size=8, color='red')
                        ))
            
            fig.update_layout(
                title=f'Switching Energy vs Current: {dev_name}<br>(V_dc={cb_v_dc.value}V, R_g={cb_r_g.value}Ω)',
                xaxis_title='Load Current (A)',
                yaxis_title='Switching Energy (µJ)',
                width=1000,
                height=600,
                hovermode='x unified'
            )
            
            fig.show()
            
            print(f"\n✓ Current sweep complete ({len(i_range)} points)")
            
        except Exception as e:
            print(f"\n⚠ Error: {e}")
            import traceback
            traceback.print_exc()

sweep_run_button.on_click(run_current_sweep)

display(widgets.VBox([
    widgets.HTML("<h4>Energy vs Current Sweep</h4>"),
    sweep_i_min,
    sweep_i_max,
    sweep_n_points,
    sweep_device_selector,
    sweep_run_button,
    sweep_output
]))

## 6. Export & Save

Export filtered device list and analysis results.

In [ ]:
# Export filtered list to CSV
export_filename = widgets.Text(
    value='filtered_devices.csv',
    description='Filename:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px')
)

export_button = widgets.Button(
    description='Export to CSV',
    button_style='success',
    icon='download'
)

export_output = widgets.Output()

def export_csv(b):
    with export_output:
        clear_output(wait=True)
        try:
            output_path = Path.cwd() / export_filename.value
            df_filtered.to_csv(output_path, index=False)
            print(f"✓ Exported {len(df_filtered)} devices to: {output_path}")
        except Exception as e:
            print(f"⚠ Export failed: {e}")

export_button.on_click(export_csv)

display(widgets.VBox([
    widgets.HTML("<h3>Export Filtered Device List</h3>"),
    export_filename,
    export_button,
    export_output
]))

## 7. Summary

This dashboard provides:

✓ **Database overview** with statistics on 245 devices  
✓ **Interactive filtering** by device type, voltage, current, manufacturer  
✓ **Performance plots**: FOM scatter, V-I rating, capacitance curves, I-V characteristics  
✓ **Christen-Biela analytical model** for switching loss prediction  
✓ **Current sweep** with model vs measured data comparison  
✓ **CSV export** for filtered device lists

### Usage Tips:

1. **Start with filters**: Configure device type, voltage class, and data requirements, then click "Apply Filters"
2. **Explore plots**: Use interactive Plotly plots to zoom, pan, and hover for details
3. **Run analytical model**: Select a device with good data coverage (capacitance + channel data) for accurate predictions
4. **Compare results**: Current sweep overlays measured data (if available) with analytical predictions
5. **Export data**: Save filtered lists for further analysis in spreadsheets or other tools

### Key Features:

- **Capacitance overlay**: Compare C_oss(V), C_iss(V), C_rss(V) across multiple devices
- **ChristenBielaModel**: State-of-the-art analytical switching loss model (2019 IEEE TPEL)
- **Data completeness filtering**: Find devices with specific measurement data
- **Interactive widgets**: Real-time parameter adjustment without re-running cells

---

**Note**: This notebook uses the `transistordatabase` package. For questions or issues, refer to the [repository documentation](https://github.com/upb-lea/transistordatabase).